# Popular and random retrieval
In this notebook, we'll implement 2 simple baselines, which are popular and ramdom retrieval. This baseline is an anchor point for developing more complex models.

### Set up

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
import warnings
from datetime import datetime, timedelta
from typing import List

import datasets
import numpy as np
import pandas as pd
import plotly.express as px
from datasets import load_dataset
from dotenv import load_dotenv
from evidently.metrics import (
    FBetaTopKMetric,
    MAPKMetric,
    MRRKMetric,
    NDCGKMetric,
    NoveltyMetric,
    PersonalizationMetric,
    PrecisionTopKMetric,
    RecallTopKMetric,
    RecCasesTable
)
from evidently.pipeline.column_mapping import ColumnMapping
from evidently.report import Report
from loguru import logger
from pydantic import BaseModel
from tqdm.notebook import tqdm

sys.path.insert(0, "..")

from src.visualization.setup import color_scheme
from src.utils.hash import hash_string_to_int
from src.eval.popularity_bias import PopularityBias
load_dotenv()
datasets.logging.set_verbosity_error()
warnings.filterwarnings(
    action="ignore",
    category=FutureWarning,
    module=r"evidently.metrics.recsys.precision_recall_k",
)

In [3]:
class Args(BaseModel):
    testing: bool = False
    log_to_mlflow: bool = True
    run_name: str = "010-MVP-basic-retrieval"
    experiment_name: str = "Random and Popular Retrieval Baseline"
    hf_dataset_path: str = "McAuley-Lab/Amazon-Reviews-2023"
    notebook_persist_dir: str = None
    random_seed: int = 41
    train_data_path: str = "../data/interim/train.parquet"
    val_data_path:str = "../data/interim/val.parquet"
    test_data_path:str = "../data/interim/test.parquet"
    
    def init(self):
        self.notebook_persist_dir = os.path.abspath(f"data/{self.run_name}")

        if not os.environ.get("MLFLOW_TRACKING_URI"):
            logger.warning(
                f"Environment variable MLFLOW_TRACKING_URI is not set. Setting self.log_to_mlflow to false."
            )
            self.log_to_mlflow = False

        if self.log_to_mlflow:
            logger.info(
                f"MLflow experiment {self.experiment_name} - run {self.run_name}..."
            )
        return self


args = Args().init()

print(args.model_dump_json(indent=2))

2025-01-08 00:55:47.800 | INFO     | __main__:init:23 - MLflow experiment Random and Popular Retrieval Baseline - run 010-MVP-basic-retrieval...
{
  "testing": false,
  "log_to_mlflow": true,
  "run_name": "010-MVP-basic-retrieval",
  "experiment_name": "Random and Popular Retrieval Baseline",
  "hf_dataset_path": "McAuley-Lab/Amazon-Reviews-2023",
  "notebook_persist_dir": "/home/dinhln/Desktop/MLOPS/recsys/HM-ScalableRecs/notebooks/data/010-MVP-basic-retrieval",
  "random_seed": 41,
  "train_data_path": "../data/interim/train.parquet",
  "val_data_path": "../data/interim/val.parquet",
  "test_data_path": "../data/interim/test.parquet"
}


In [4]:
# Load data (just need train and val for now)
%time
train_df = pd.read_parquet(args.train_data_path)
val_df = pd.read_parquet(args.val_data_path, columns=['customer_id', 'article_id'])

CPU times: user 2 μs, sys: 0 ns, total: 2 μs
Wall time: 5.01 μs


In [5]:
train_df.head()

,customer_id,event_timestamp,article_id
393,a36ab8958f1eb3e65bd2299316b0c8a17a70eb5fbbbd52...,2020-03-02,372860002
396,a36ab8958f1eb3e65bd2299316b0c8a17a70eb5fbbbd52...,2020-03-02,372860001
1067,d89d7baaab263672a0e05330f6dcbd92211074420af078...,2020-03-04,852174001
1068,d89d7baaab263672a0e05330f6dcbd92211074420af078...,2020-03-04,852174001
1071,d89d7baaab263672a0e05330f6dcbd92211074420af078...,2020-03-04,841228001


In [6]:
val_df.head()

,customer_id,article_id
50209,e4f304fc282ab470cd1e3e81671f6658bf39e1b774feea...,823505002
50210,e4f304fc282ab470cd1e3e81671f6658bf39e1b774feea...,806225008
50211,e4f304fc282ab470cd1e3e81671f6658bf39e1b774feea...,868018003
50212,e4f304fc282ab470cd1e3e81671f6658bf39e1b774feea...,699081001
53156,ebc0d2e437e43aeb59aa55c271c90f2a6e8157d9be33b6...,800691007


## Popular retrieval

In [7]:
class CommonModelParams(BaseModel):
    top_K: int = 100
    top_k: int = 10
    num_epochs: int = 100

    compare_with_random: bool = True


cmparams = CommonModelParams()

In [8]:
popular_item_df = (
    train_df.groupby("article_id", as_index = False)
    .size()
    .assign(
        rec_ranking = lambda df: df["size"].rank(method = "first", ascending= False).astype(int)
    )
    .sort_values(["rec_ranking"], ascending= [True])
    .head(cmparams.top_K)
)
popular_item_df

,article_id,size,rec_ranking
65,599580038,69,1
134,706016001,58,2
70,599580052,48,3
81,610776002,48,4
198,741356002,45,5
...,...,...,...
225,759465001,19,96
227,759482001,19,97
239,766346003,19,98
252,776237006,19,99


In [9]:
popular_recs = (
    val_df[["customer_id"]]
    .drop_duplicates()
    .assign(key = 1)
    .merge(popular_item_df.assign(key = 1), on = "key", how = "left")
    .rename(columns = {"size": "score"})
    .drop(columns = ["key"])
)
popular_recs

,customer_id,article_id,score,rec_ranking
0,e4f304fc282ab470cd1e3e81671f6658bf39e1b774feea...,599580038,69,1
1,e4f304fc282ab470cd1e3e81671f6658bf39e1b774feea...,706016001,58,2
2,e4f304fc282ab470cd1e3e81671f6658bf39e1b774feea...,599580052,48,3
3,e4f304fc282ab470cd1e3e81671f6658bf39e1b774feea...,610776002,48,4
4,e4f304fc282ab470cd1e3e81671f6658bf39e1b774feea...,741356002,45,5
...,...,...,...,...
22095,d20d5215208b1d6c76d79b741d037b8945f58e80d6d8eb...,759465001,19,96
22096,d20d5215208b1d6c76d79b741d037b8945f58e80d6d8eb...,759482001,19,97
22097,d20d5215208b1d6c76d79b741d037b8945f58e80d6d8eb...,766346003,19,98
22098,d20d5215208b1d6c76d79b741d037b8945f58e80d6d8eb...,776237006,19,99


In [10]:
popular_recs.to_csv(f"{args.notebook_persist_dir}/popular_recs.csv", index = False)

## Random retrieval

In [11]:
import uuid
all_unique_items = train_df["article_id"].unique()

def get_random_item(customer_id: str, all_items: List[str]) -> List[str]:
    np.random.seed(hash_string_to_int(customer_id))
    return list(np.random.choice(all_items, cmparams.top_K, replace = False))

In [12]:
if cmparams.compare_with_random:
    random_recs_list = []
    for customer_id in tqdm(val_df["customer_id"].unique(), total=val_df["customer_id"].nunique()):
        uid_random_recs = get_random_item(customer_id, all_unique_items)
        uid_random_recs_df = (
            pd.DataFrame(uid_random_recs, columns=["article_id"])
            .reset_index()
            .assign(
                customer_id=customer_id,
                index=lambda df: df["index"] + 1,
                score=lambda df: 1 / df["index"],
            )
            .rename(columns={"index": "rec_ranking"})[
                ["customer_id", "article_id", "score", "rec_ranking"]
            ]
        )
        random_recs_list.append(uid_random_recs_df)
    random_recs_df = pd.concat(random_recs_list, axis=0)


  0%|          | 0/221 [00:00<?, ?it/s]

In [13]:
random_recs_df

,customer_id,article_id,score,rec_ranking
0,e4f304fc282ab470cd1e3e81671f6658bf39e1b774feea...,821163003,1.000000,1
1,e4f304fc282ab470cd1e3e81671f6658bf39e1b774feea...,690936006,0.500000,2
2,e4f304fc282ab470cd1e3e81671f6658bf39e1b774feea...,783346016,0.333333,3
3,e4f304fc282ab470cd1e3e81671f6658bf39e1b774feea...,640021011,0.250000,4
4,e4f304fc282ab470cd1e3e81671f6658bf39e1b774feea...,684209019,0.200000,5
...,...,...,...,...
95,d20d5215208b1d6c76d79b741d037b8945f58e80d6d8eb...,554598047,0.010417,96
96,d20d5215208b1d6c76d79b741d037b8945f58e80d6d8eb...,854677002,0.010309,97
97,d20d5215208b1d6c76d79b741d037b8945f58e80d6d8eb...,796210001,0.010204,98
98,d20d5215208b1d6c76d79b741d037b8945f58e80d6d8eb...,902107001,0.010101,99


In [14]:
random_recs_df.to_csv(f"{args.notebook_persist_dir}/random_recs.csv", index = False)

## Evaluate with Evidently

In [15]:
def merge_recs_with_target(recs_df, label_df):
    return (
        recs_df.pipe(
            lambda df: pd.merge(
                df,
                label_df,
                on=["customer_id", "article_id"],
                how="outer",
            )
        )
        .assign(
            target = lambda df: df["target"].fillna(0).astype(int),
            # Fill the recall with ranking = top_K + 1 so that the recall calculation is correct
            rec_ranking=lambda df: df["rec_ranking"]
            .fillna(cmparams.top_K + 1)
            .astype(int),
        )
        .sort_values(["customer_id", "rec_ranking"])
    )

In [16]:
label_df = val_df.assign(target = 1)


In [17]:
popular_recs_target = merge_recs_with_target(popular_recs, label_df)
popular_recs_target

,customer_id,article_id,score,rec_ranking,target
10,01f88d6d339856650c616b3678793a4e6e241d87fc3a3d...,599580038,69.0,1,0
28,01f88d6d339856650c616b3678793a4e6e241d87fc3a3d...,706016001,58.0,2,0
12,01f88d6d339856650c616b3678793a4e6e241d87fc3a3d...,599580052,48.0,3,0
16,01f88d6d339856650c616b3678793a4e6e241d87fc3a3d...,610776002,48.0,4,0
39,01f88d6d339856650c616b3678793a4e6e241d87fc3a3d...,741356002,45.0,5,0
...,...,...,...,...,...
22508,ff871bad0007fece7af7c0e7e0d83566f80f0184cfd7bc...,505882002,NaN,101,1
22510,ff871bad0007fece7af7c0e7e0d83566f80f0184cfd7bc...,559616013,NaN,101,1
22535,ff871bad0007fece7af7c0e7e0d83566f80f0184cfd7bc...,713253003,NaN,101,1
22579,ff871bad0007fece7af7c0e7e0d83566f80f0184cfd7bc...,817472004,NaN,101,1


In [18]:
random_recs_target = merge_recs_with_target(random_recs_df, label_df)
random_recs_target

,customer_id,article_id,score,rec_ranking,target
40,01f88d6d339856650c616b3678793a4e6e241d87fc3a3d...,759871025,1.000000,1,0
55,01f88d6d339856650c616b3678793a4e6e241d87fc3a3d...,811927007,0.500000,2,0
88,01f88d6d339856650c616b3678793a4e6e241d87fc3a3d...,854683004,0.333333,3,0
75,01f88d6d339856650c616b3678793a4e6e241d87fc3a3d...,838787005,0.250000,4,0
28,01f88d6d339856650c616b3678793a4e6e241d87fc3a3d...,723529001,0.200000,5,0
...,...,...,...,...,...
22551,ff871bad0007fece7af7c0e7e0d83566f80f0184cfd7bc...,505882002,NaN,101,1
22554,ff871bad0007fece7af7c0e7e0d83566f80f0184cfd7bc...,559616013,NaN,101,1
22568,ff871bad0007fece7af7c0e7e0d83566f80f0184cfd7bc...,713253003,NaN,101,1
22610,ff871bad0007fece7af7c0e7e0d83566f80f0184cfd7bc...,817472004,NaN,101,1


In [19]:
column_mapping = ColumnMapping(
    recommendations_type="rank",
    target="target",
    prediction="rec_ranking",
    item_id="article_id",
    user_id="customer_id",
    datetime="event_timestamp",
)

fighters = {
    "current": {"name": "popular", "recs": popular_recs_target},
    "reference": {"name": "random", "recs": random_recs_target},
}

report = Report(
    metrics=[
        NDCGKMetric(k=cmparams.top_k),
        RecallTopKMetric(k=cmparams.top_K),
        PrecisionTopKMetric(k=cmparams.top_k),
        FBetaTopKMetric(k=cmparams.top_k),
        MAPKMetric(k=cmparams.top_k),
        MRRKMetric(k=cmparams.top_k),
        NoveltyMetric(k=cmparams.top_k),
        PersonalizationMetric(k=cmparams.top_k),
        PopularityBias(k=cmparams.top_k),
        RecCasesTable(item_num=cmparams.top_k),
    ],
    options=[color_scheme],
)

report.run(
    reference_data=fighters["reference"]["recs"],
    current_data=fighters["current"]["recs"],
    column_mapping=column_mapping,
    additional_data={"current_train_data": train_df,"current_name": fighters["current"]["name"], "reference_name": fighters["reference"]["name"],},
)

evidently_report_fp = f"{args.notebook_persist_dir}/{fighters['current']['name']}_vs_{fighters['reference']['name']}_evidently_report.html"
os.makedirs(args.notebook_persist_dir, exist_ok=True)
logger.info(f"Evaluation metrics logged to {evidently_report_fp}")
report.save_html(evidently_report_fp)

/home/dinhln/Desktop/MLOPS/recsys/HM-ScalableRecs/.venv/lib/python3.11/site-packages/evidently/metrics/recsys/f_beta_top_k.py:64: RuntimeWarning: invalid value encountered in divide
  return (1 + beta_sqr) * precision_arr * recall_arr / (beta_sqr * precision_arr + recall_arr)


2025-01-08 00:55:50.106 | INFO     | __main__:<module>:40 - Evaluation metrics logged to /home/dinhln/Desktop/MLOPS/recsys/HM-ScalableRecs/notebooks/data/010-MVP-basic-retrieval/popular_vs_random_evidently_report.html


### Log to MLflow

In [20]:
if args.log_to_mlflow:
    import mlflow
    import glob
    mlflow.set_experiment(args.experiment_name)
    with mlflow.start_run(run_name=args.run_name):
        mlflow.log_artifact(evidently_report_fp)
        for file_path in glob.glob(f"{args.notebook_persist_dir}/*.csv"):
            mlflow.log_artifact(file_path, artifact_path="ramdom_popular_recs")

🏃 View run 010-MVP-basic-retrieval at: http://localhost:5002/#/experiments/2/runs/18630157b7694f15b488c3df308ca421
🧪 View experiment at: http://localhost:5002/#/experiments/2
